# Poetry Through the Eyes of a Machine
An exploratory study of 12 original poems using three pretrained text embedding models. Run all cells in order. CPU is sufficient; internet access is required for model downloads. The embedded corpus is private: do not publish this notebook without reviewing permissions. AI-assisted annotations are provisional, not independent author ground truth.

In [ ]:
%pip -q install sentence-transformers scikit-learn pandas matplotlib seaborn

## 1. Load the original dataset
Only poem text is embedded; titles and annotations are used for display and exploratory comparisons.

In [ ]:
import random, json, shutil, sys, importlib.metadata as im
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from scipy.stats import spearmanr
from sklearn.feature_extraction.text import TfidfVectorizer
from IPython.display import display
random.seed(42); np.random.seed(42); torch.manual_seed(42)
OUT = Path('poetry_results'); OUT.mkdir(exist_ok=True)
records = [{'poem_id': 'P01', 'title': 'The Painter', 'poem_text': 'there was a painter\nwho once said\nthat reality is the least honest thing in the world.\n\neveryone laughed at him for that.\n\nbecause reality is measurable, isn’t it?\ntouchable. visible.\nit exists in definite shapes\nand names that stay still long enough\nfor language to hold them.\n\nbut he kept thinking of all the things\nthat had changed him completely\nwithout ever becoming visible at all.\n\nhow do you paint the feeling\nof returning home\nand realising you no longer belong to it\nin quite the same way?\n\nhow do you paint the exact moment\nsomeone stops being a person you know\nand quietly becomes part\nof the architecture of your inner world?\n\nhow do you paint longing\nbefore it learns its own name?\n\nhe spent years trying.\n\nnot painting people…\npeople were easy.\nfaces are only geometry.\nlight resting temporarily on bone.\n\nhe wanted to paint the invisible movements beneath them.\n\nthe unbearable tenderness\nof someone remembering the smallest thing about you.\n\nthe strange ache\nof watching time pass through another person’s face.\n\nthe way grief does not arrive screaming,\nbut instead slips silently into ordinary objects;\na half-read book,\nan untouched cup of coffee,\na chair angled slightly toward the door\nas if still waiting.\n\nhe painted in impossible colors.\n\nnot red,\nbut the color of almost becoming.\n\nnot blue,\nbut the color of memories\nthat feel more alive\nthan the moments themselves ever did.\n\npeople looked at his canvases and said,\n“this doesn’t look real.”\n\nand he always wanted to ask them…\n\nwas the trembling in your chest real\nthe first time you realized\nanother human being\nhad somehow altered the gravity inside you?\n\nis music unreal\nbecause you cannot hold it in your hands?\n\nis a dream less true\nsimply because morning survives it?\n\nhe believed truth lived elsewhere.\n\nnot in objects themselves,\nbut in the silent spaces surrounding them.\n\nin pauses during conversations.\nin the unbearable intimacy of being understood accidentally.\nin the silence after laughter fades,\nwhen you suddenly become aware\nof how temporary every beautiful thing is.\n\nthat was what he painted.\n\nnot reality itself,\nbut the wound reality leaves behind\nwhen it touches the soul deeply enough.\n\nand one night,\nafter years of failure,\nhe painted something that frightened him.\n\nit was not a person.\nnot a landscape.\nnot anything recognizable.\njust light\nfalling through a half open door\ninto an empty room.\n\nbut somehow,\ninside that room\nlived every goodbye he had never recovered from,\nevery version of himself\nhe had abandoned to survive,\nevery fleeting beautiful thing\nhe had loved too deeply\nto keep.\n\nhe stared at the canvas for hours afterward.\n\nbecause for the first time in years,\nhe had not painted what something looked like.\n\nhe had painted\nwhat it meant to lose it.\n\nand perhaps that is the closest art ever comes\nto telling the truth…\n\nnot by reproducing the world faithfully,\nbut by revealing the invisible devastations\nrequired to survive it.\n\nbecause the truth does not live\nin reality itself.\n\nit lives in the place\nwhere reality ends\nand feeling begins.', 'title_is_working': True, 'date_written': '', 'author_intent': 'I wanted to explore the difference between reproducing what is visible and expressing what an experience actually means. The painter finds truth in the emotional traces that ordinary objects and moments carry, rather than in their outward appearance.', 'author_themes': 'reality; emotional truth; representation; memory; loss', 'source_note': '', 'annotation_source': 'AI-assisted draft inferred from poem text', 'author_review_status': 'pending', 'related_poem_ids': 'P05;P09', 'relationship_source': 'AI-inferred candidate relationships; not author validated'}, {'poem_id': 'P02', 'title': 'These Mountains', 'poem_text': "these mountains that you are holding,\nyou were only supposed to climb.\n\nthey were never meant for your shoulders,\nthey were meant for your steps.\n\nfor the shaky knees,\nthe lungs that burn,\nthe soul that feels lighter each time.\n\ni thought maybe the ache was proof of living—\nbut no, love,\nyou weren't born to collapse under the sky.\n\nyou were born to touch it.\n\nso set it down.\nbreathe.\njust climb.", 'title_is_working': True, 'date_written': '', 'author_intent': 'I wanted to distinguish facing difficulties from carrying them as permanent burdens. Struggling is not the purpose of living; the mountain is something to move through, not something that must define or exhaust us.', 'author_themes': 'burden; perseverance; release; self-compassion', 'source_note': '', 'annotation_source': 'AI-assisted draft inferred from poem text', 'author_review_status': 'pending', 'related_poem_ids': 'P03;P08;P12', 'relationship_source': 'AI-inferred candidate relationships; not author validated'}, {'poem_id': 'P03', 'title': 'Forward Is Forward', 'poem_text': 'forward is forward,\neven when it feels like nothing changed today.\nyou are still moving,\nstill weaving little threads of tomorrow\nwith every breath you take.\n\nthe river doesn’t rush all the time—\nsometimes it drifts,\nsometimes it rests,\nbut it always flows.\n\nso do you.\n\nyou don’t need to hurry.\nyou don’t need to prove.\nyou are enough in your becoming,\nenough in your slow and steady light.\n\nrest in that truth:\nyou are on your way,\nand that is more than enough.', 'title_is_working': True, 'date_written': '', 'author_intent': 'I wanted to make room for progress that is slow or invisible. Rest and uncertainty do not necessarily mean that we have stopped growing, and our worth does not depend on constantly proving ourselves.', 'author_themes': 'patience; growth; rest; self-worth', 'source_note': '', 'annotation_source': 'AI-assisted draft inferred from poem text', 'author_review_status': 'pending', 'related_poem_ids': 'P02;P08', 'relationship_source': 'AI-inferred candidate relationships; not author validated'}, {'poem_id': 'P04', 'title': 'Broken Glass', 'poem_text': 'she didn’t just pick up the pieces,\nshe bled on them,\nshe carved her pain into them,\nand built a mosaic the world had never seen.\n\nthe world called her broken,\nbut they didn’t know—\nbroken glass reflects more light\nthan something whole ever could.\n\nshe was no longer a girl piecing herself together,\nshe was the fire that rose from her own ashes.\nand when she walked,\nthe air shifted—\nfor she carried a new truth:\nbroken isn’t the end,\nbroken is the beginning.', 'title_is_working': True, 'date_written': '', 'author_intent': 'I wanted to portray recovery as transformation rather than a return to an earlier, undamaged self. The mosaic represents making something new from painful experiences without pretending those experiences never happened.', 'author_themes': 'recovery; transformation; resilience; identity', 'source_note': '', 'annotation_source': 'AI-assisted draft inferred from poem text', 'author_review_status': 'pending', 'related_poem_ids': 'P07;P08', 'relationship_source': 'AI-inferred candidate relationships; not author validated'}, {'poem_id': 'P05', 'title': 'rooms that remember truth', 'poem_text': '.\nthere’s a kind of magic in walking slow,\nthrough halls where the light feels older than the walls.\nit lands in soft pools\non leather chairs,\ndusty book edges,\nhalf-sipped tea.\neverything still.\neverything listening.\n.\nthe paintings don’t speak—\nbut they watch.\nand in that watching,\nyou realize:\nsome things don’t need words\nto be understood.\n.\nthere’s a book that’s been waiting for me\nfor years.\nnot because it holds answers—\nbut because it holds\nthe questions\ni’m finally ready to ask.\n.\nand maybe that’s what depth is.\nnot something you dig into,\nbut something you grow beside.\nlike how silence can be full.\nor how a moment can stretch\nand still feel complete.\n.\nin these quiet corners,\ni feel like the world stops measuring me.\nnot by how much i do,\nbut by how much i notice.\nhow long i can sit with a thought\nwithout running away.\n.\ntruth doesn’t always knock.\nsometimes, it just\nwaits.\n.\nand you find it\nin a dusty bookstore,\nin the gold frame of a stranger’s portrait,\nin the way a page turns slowly—\nnot revealing,\nbut remembering.', 'title_is_working': False, 'date_written': '', 'author_intent': 'I wanted to explore how stillness makes us receptive to beauty, understanding and self-awareness. Sometimes we become ready for a question rather than finding an answer, and noticing can matter more than accomplishing.', 'author_themes': 'stillness; attention; self-awareness; curiosity; truth', 'source_note': 'true understanding, beauty, and self-awareness come in stillness — not from seeking answers, but from being open to the quiet moments that hold them.', 'annotation_source': 'AI-assisted draft inferred from poem text', 'author_review_status': 'pending', 'related_poem_ids': 'P01;P10;P11', 'relationship_source': 'AI-inferred candidate relationships; not author validated'}, {'poem_id': 'P06', 'title': 'and the canary kept singing', 'poem_text': 'what if the spear missed?\n\nwhat if the gods blinked —\njust once —\nand the little bird slipped through their fingers?\n\nrue lives.\n\nnot untouched.\nnot unharmed.\nbut changed.\n\nsomething in her eyes turns glassy,\nlike she’s seen past the veil\nand returned with silence stitched beneath her tongue.\n\nkatniss finds her in the hollow of a tree,\nblood dried like rust along her arms,\nstill singing —\na lullaby with no melody,\njust breath and tremble and grief.\n\nthey move through the arena like ghosts.\nnot hunted.\nhaunting.\n\nrue speaks less now.\nonly in riddles.\nonly in dreams.\n\n“they built this place on bones,” she whispers,\n“so it will starve when we give it none.”\n\nthey starve the game.\nrefuse violence.\nrefuse story.\n\nthey smear mud on the cameras,\nstring up dolls from branches,\ntear the eyes from the trees.\n\npanem watches.\nthe capitol seethes.\n\nthese are not girls.\nthey are omens.\n\nand when the final order comes —\n“one must die” —\nthey do not flinch.\n\nrue walks to the center.\nbarefoot.\nno weapon.\nno tears.\n\n“you want blood?”\nher voice cracks like a hymn.\n“then choke on mine.”\n\nshe opens her arms.\ncalls down fire.\n\nthey call it a system failure.\na broadcast glitch.\na technical tragedy.\n\nbut the districts know.\n\nthey saw the girl who lived\nwalk into the flames\nwith a smile that didn’t belong in this world.\n\nkatniss is crowned.\nbut her eyes are empty.\nshe dreams of twisted roots,\na songbird without a beak,\nand smoke that tastes like rue’s breath.\n\nthe games ended.\nbut something older woke up that day.\nand it sings still —\nlow, cold, and vengeful\nin every crack of the stone.', 'title_is_working': False, 'date_written': '', 'author_intent': 'I wanted to imagine Rue surviving and changing the rules of the Games through refusal and resistance. Her eventual sacrifice exposes the violence of a system that depends on turning suffering into spectacle.', 'author_themes': 'resistance; sacrifice; exploitation; grief; agency', 'source_note': '', 'annotation_source': 'AI-assisted draft inferred from poem text', 'author_review_status': 'pending', 'related_poem_ids': 'P07', 'relationship_source': 'AI-inferred candidate relationships; not author validated'}, {'poem_id': 'P07', 'title': 'the orchard fed her bones', 'poem_text': '.\nthey said the girl died clean.\n.\nthey lied.\n.\nrue did not die.\nrue was taken.\nswallowed by the trees,\nstitched back together with rot and root,\nreassembled in the dark belly of the arena —\na girl reborn, not in grace, but in vengeance.\n.\nwhen katniss finds her,\nshe is not whole.\nher eyes are too black,\ntoo still.\nher voice, when it comes, sounds like something buried too long.\n.\n"they tried to harvest me,” she says,\n“but the orchard took me first.”\n.\n.\n.\nshe bleeds sap.\nshe sings in tongues.\nshe moves like something that remembers being human\nbut isn’t anymore.\n.\nkatniss follows her anyway.\nof course she does.\neven when the air turns sour,\neven when the trees whisper names not spoken in decades,\neven when the ground breathes beneath their feet.\n.\nthe arena begins to decay.\n.\nmoss spreads like rot across the cornucopia.\ncameras crack.\nmutts gnaw at their own paws.\nthe sky weeps black rain.\n.\nand rue moves through it all, untouched,\na child-shaped plague.\n.\nthe Capitol sends in fire.\n.\nrue laughs.\nit is the sound of teeth on bone.\n.\n“they want a show?”\nher smile is a wound.\n“then let them choke on ruin.”\n.\n.\n.\nshe does not die.\nshe unravels.\n.\nat the final countdown, when only two remain,\nshe walks to the center of the arena\nand opens her mouth wide.\n.\nfrom her throat pours ash,\ncarrion birds,\nfragments of hymns older than the Capitol itself.\n.\nthe screen goes black.\nno victor.\nno anthem.\n.\njust silence.\n.\nin the weeks that follow,\ndistrict children begin waking with soil under their nails.\nthe woods grow wrong.\nmockingjays fall from the sky mid-song.\nand in the corners of mirrors,\nthey see her —\nsmall, barefoot, watching.\n.\nstill humming.', 'title_is_working': False, 'date_written': '', 'author_intent': 'I wanted to imagine a darker version of Rue surviving, where the arena transforms her into something that turns its violence against its creators. The horror represents the consequences of exploitation returning to haunt the system.', 'author_themes': 'vengeance; transformation; exploitation; horror; resistance', 'source_note': 'an alternate ending if rue lived', 'annotation_source': 'AI-assisted draft inferred from poem text', 'author_review_status': 'pending', 'related_poem_ids': 'P04;P06', 'relationship_source': 'AI-inferred candidate relationships; not author validated'}, {'poem_id': 'P08', 'title': 'the phoenix knows', 'poem_text': "the phoenix knows\nthat burning isn't always violent.\nsometimes, it's quiet—\na slow undoing,\na surrender to gravity\nbefore the sky remembers your name.\n.\nshe doesn’t rise for glory.\nshe rises because there’s nothing left\nbut the choice\nto begin again.\n.\nash is not the end.\nit is memory—\nof what was heavy,\nwhat had to fall apart.\n.\nand from it,\nshe gathers herself—\nnot new,\nbut changed.\n.\nnot untouched,\nbut honest.\n.\neach feather a story\nof what the fire took,\nand what it couldn’t.\n.\nthe phoenix never asks\nto be seen.\nshe only asks\nto survive.\nand still,\nshe rises.", 'title_is_working': False, 'date_written': '2025-06-30', 'author_intent': 'I wanted to portray survival as something quieter than a triumphant comeback. The phoenix carries the memory of what happened and rises because beginning again is the only choice left, not because she wants recognition.', 'author_themes': 'survival; rebirth; memory; quiet resilience', 'source_note': '', 'annotation_source': 'AI-assisted draft inferred from poem text', 'author_review_status': 'pending', 'related_poem_ids': 'P02;P03;P04', 'relationship_source': 'AI-inferred candidate relationships; not author validated'}, {'poem_id': 'P09', 'title': 'wax child', 'poem_text': 'i saw her hold him like he was whole —\na mother and son, a stitched-up soul.\nthey walked through noise,\nbut she only heard\nhis laugh,\nhis little word.\n\nthen —\na scream in steel.\na shot, too real.\nher leg caught fire\nbut her voice stayed calm.\nshe only ached\nfor her baby’s palm.\n\nhe stood so still,\nin the heart of it all,\nwatched her stumble,\nwatched her fall.\na tear slipped down —\nand he began to fade.\nhis skin ran soft.\nhis shape unmade.\n\nnot a boy.\nnever one.\njust wax beneath a dying sun.\nand still,\nshe reached.\nshe called his name\nlike love alone\ncould stop the flame.\n\nhe broke.\nlimbs cracked,\ncold and light.\nand she held him close\nlike it was all right.\n\nthey said\n“he was never real.”\nbut what do they know\nof the way we feel?\n\nshe rocked him slow\nwith empty hands,\nstill humming songs\nhe’d never understand.\n\nand when they pulled\nher grief away,\nher arms were shaped\nfor him to stay.', 'title_is_working': False, 'date_written': '', 'author_intent': 'I wanted to question whether love and grief become less real when the object of attachment is not conventionally alive. The mother’s attachment remains meaningful even as the child’s physical form disappears.', 'author_themes': 'attachment; grief; love; perceived reality; loss', 'source_note': '', 'annotation_source': 'AI-assisted draft inferred from poem text', 'author_review_status': 'pending', 'related_poem_ids': 'P01;P06', 'relationship_source': 'AI-inferred candidate relationships; not author validated'}, {'poem_id': 'P10', 'title': 'It Starts With Thought', 'poem_text': "Before the hand can draw a line,\nbefore the stars are made to shine,\nbefore the world can call it mine—\na thought must whisper, soft, divine.\n\nIt doesn't shout, it doesn’t show,\nit plants its seeds and lets them grow.\nWhat blooms in time, the world will know—\nbut roots run deep where thoughts first go.\n\nA moment’s pause, a silent spark,\ncan shift the day, can light the dark.\nThe smallest drift, the slightest mark—\ncan leave behind a lasting arc.\n\nSo guard your mind, both wild and wide,\nfor that’s the place where truths decide.\nAll that you do, and what you hide—\nbegan with thought, not step or stride.", 'title_is_working': False, 'date_written': '', 'author_intent': 'I wanted to reflect on the invisible beginnings of creation and action. A thought may seem insignificant, but it can shape choices and eventually leave a lasting effect on the world.', 'author_themes': 'thought; creation; agency; reflection', 'source_note': '', 'annotation_source': 'AI-assisted draft inferred from poem text', 'author_review_status': 'pending', 'related_poem_ids': 'P05;P11', 'relationship_source': 'AI-inferred candidate relationships; not author validated'}, {'poem_id': 'P11', 'title': 'The Knowing Quiet', 'poem_text': "Listen to the wind—it speaks in sighs,\nin dancing leaves and open skies.\nIt tells of things you used to feel,\nbefore the world taught you to kneel.\n\nListen to the silence—it holds space,\nfor every tear you didn't face.\nIt doesn't judge, it doesn't flee,\nit simply waits for you to see.\n\nListen to your heart—it knows the way,\nthrough broken nights and heavy days.\nIt may be soft, but it is strong—\nit hums the truth when all feels wrong.\n\nThere is a voice beneath the noise,\nbeyond the crowds, beyond the choice.\nIt lives in stillness, breath, and flame—\nand when you hear it, you're not the same.", 'title_is_working': False, 'date_written': '', 'author_intent': 'I wanted to explore the quieter parts of ourselves that become easier to notice away from external noise and expectations. Silence can make room for emotion and self-understanding.', 'author_themes': 'stillness; intuition; introspection; emotional awareness', 'source_note': '', 'annotation_source': 'AI-assisted draft inferred from poem text', 'author_review_status': 'pending', 'related_poem_ids': 'P05;P10', 'relationship_source': 'AI-inferred candidate relationships; not author validated'}, {'poem_id': 'P12', 'title': 'Call of the Brave', 'poem_text': 'There was never a promise of safety,\nonly the whisper of stars beyond sight—\na voice in the wind that said go,\nthat dared the heart to ignite.\n\nThe path was not paved with answers,\nnor lit by lanterns of ease;\nit twisted through thunder and silence,\nthrough forests that brought you to knees.\n\nBut those who walked it with wonder,\nwith fire in marrow and bone,\nfound treasures not made of silver—\nbut courage, carved deep in their own.\n\nSo cast off the cloak of comfort,\nstep past the walls of known.\nFor life is not meant to be measured—\nbut tasted, and risked, and grown.', 'title_is_working': False, 'date_written': '', 'author_intent': 'I wanted to portray courage as moving beyond certainty rather than being guaranteed safety. The value of an unfamiliar journey lies partly in how it changes us, rather than in what we acquire.', 'author_themes': 'courage; uncertainty; exploration; personal growth', 'source_note': '', 'annotation_source': 'AI-assisted draft inferred from poem text', 'author_review_status': 'pending', 'related_poem_ids': 'P02;P03', 'relationship_source': 'AI-inferred candidate relationships; not author validated'}]
df = pd.DataFrame(records)
assert len(df)==12 and df.poem_id.is_unique and df.poem_text.str.strip().ne('').all()
titles=df.title.tolist(); ids=df.poem_id.tolist()
df['word_count']=df.poem_text.str.split().str.len()
display(df[['poem_id','title','word_count','author_review_status']])
meta={'python':sys.version,'packages':{k:im.version(k) for k in ['sentence-transformers','transformers','torch','scikit-learn','numpy','pandas']}}


## 2. Models and length-aware embeddings
Split long texts into non-overlapping token-budgeted spans using original character offsets. Average normalized chunk embeddings weighted by token count, then normalize again. This avoids silent truncation but can lose cross-chunk context. Different tokenizers may split differently.

In [ ]:
MODEL_IDS={'MiniLM':'sentence-transformers/all-MiniLM-L6-v2','MPNet':'sentence-transformers/all-mpnet-base-v2','BGE':'BAAI/bge-base-en-v1.5'}
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
models={k:SentenceTransformer(v,device=DEVICE) for k,v in MODEL_IDS.items()}
meta['models']={k:{'id':MODEL_IDS[k],'max_seq_length':m.max_seq_length,'dimensions':m.get_sentence_embedding_dimension(),'revision':getattr(m[0].auto_model.config,'_commit_hash',None)} for k,m in models.items()}
print(DEVICE,meta['models'])

def embed(text, model, prefix=''):
    tok=model.tokenizer
    budget=model.max_seq_length-tok.num_special_tokens_to_add(pair=False)
    offsets=tok(text,add_special_tokens=False,return_offsets_mapping=True,truncation=False,verbose=False)['offset_mapping']
    if not offsets: raise ValueError('Empty text')
    chunks=[]; weights=[]; i=0
    while i<len(offsets):
        j=min(i+budget,len(offsets))
        while j>i:
            span=text[offsets[i][0]:offsets[j-1][1]]
            count=len(tok.encode(prefix+span,add_special_tokens=True,truncation=False,verbose=False))
            if count<=model.max_seq_length: break
            j-=1
        if j==i: raise ValueError('Unable to fit chunk')
        chunks.append(prefix+span); weights.append(j-i); i=j
    v=model.encode(chunks,normalize_embeddings=True,show_progress_bar=False)
    mean=np.average(v,axis=0,weights=weights)
    return mean/max(np.linalg.norm(mean),1e-12),len(offsets),len(chunks)

vectors={}; matrices={}; stats=[]
for name,m in models.items():
    rows=[]
    for r in df.itertuples():
        v,t,c=embed(r.poem_text,m); rows.append(v)
        stats.append(dict(model=name,poem_id=r.poem_id,tokens=t,chunks=c))
    vectors[name]=np.stack(rows); matrices[name]=vectors[name]@vectors[name].T
    np.save(OUT/f'{name}_embeddings.npy',vectors[name])
    print(name,vectors[name].shape)
pd.DataFrame(stats).to_csv(OUT/'chunking.csv',index=False)
display(pd.DataFrame(stats))


## 3. Similarity, nearest pairs and maps
Cosine similarity is not a probability. PCA plots are approximations; use original vectors for rankings.

In [ ]:
pairs=[]
for name,S in matrices.items():
    table=pd.DataFrame(S,index=titles,columns=titles)
    table.to_csv(OUT/f'{name}_similarity.csv')
    plt.figure(figsize=(13,10)); sns.heatmap(table,annot=True,fmt='.2f',cmap='mako',vmin=-1,vmax=1,square=True)
    plt.title(f'Poetry similarity — {name}'); plt.xticks(rotation=55,ha='right'); plt.tight_layout(); plt.savefig(OUT/f'{name}_heatmap.png',dpi=220); plt.show(); plt.close()
    for i,j in combinations(range(len(df)),2):
        pairs.append(dict(model=name,poem_1=ids[i],poem_2=ids[j],title_1=titles[i],title_2=titles[j],similarity=float(S[i,j])))
    pca=PCA(n_components=2); xy=pca.fit_transform(vectors[name]); var=pca.explained_variance_ratio_*100
    plt.figure(figsize=(10,7)); plt.scatter(xy[:,0],xy[:,1],s=70)
    for i,t in enumerate(ids): plt.annotate(t,xy[i],xytext=(5,5),textcoords='offset points')
    plt.xlabel(f'PC1 ({var[0]:.1f}%)'); plt.ylabel(f'PC2 ({var[1]:.1f}%)'); plt.title(f'Poetry map — {name}'); plt.tight_layout(); plt.savefig(OUT/f'{name}_map.png',dpi=220); plt.show(); plt.close()
    pd.DataFrame(dict(poem_id=ids,title=titles,PC1=xy[:,0],PC2=xy[:,1])).to_csv(OUT/f'{name}_pca.csv',index=False)
pair_df=pd.DataFrame(pairs); pair_df.to_csv(OUT/'pairs.csv',index=False)
for name in models:
    print(name); display(pair_df.query('model == @name').nlargest(10,'similarity'))
display(df[['poem_id','title']])


## 4. Model agreement and lexical baseline
Compare rankings with TF-IDF as a simple word-overlap baseline. Correlations are descriptive, not significance tests.

In [ ]:
tf=TfidfVectorizer(ngram_range=(1,2)).fit_transform(df.poem_text)
all_matrices={**matrices,'TF-IDF':(tf@tf.T).toarray()}
tri=np.triu_indices(len(df),1)
agreement=pd.DataFrame([dict(model_1=a,model_2=b,spearman=float(spearmanr(all_matrices[a][tri],all_matrices[b][tri]).statistic)) for a,b in combinations(all_matrices,2)])
display(agreement); agreement.to_csv(OUT/'agreement.csv',index=False)
# Compare provisional AI-inferred links, not independent human labels.
links=set()
for r in df.itertuples():
    for other in r.related_poem_ids.split(';'):
        other=other.strip()
        if other in ids and other!=r.poem_id: links.add(tuple(sorted([r.poem_id,other])))
comparison=pair_df.copy(); comparison['draft_link']=[tuple(sorted([a,b])) in links for a,b in zip(comparison.poem_1,comparison.poem_2)]
comparison['rank']=comparison.groupby('model').similarity.rank(ascending=False,method='min').astype(int)
comparison.to_csv(OUT/'draft_relationships.csv',index=False)
display(comparison[comparison.draft_link].sort_values(['model','rank']))


## 5. Metaphor challenge
Each original excerpt is paired with an AI-written intended-meaning paraphrase and a literal/vocabulary-related alternative. These are illustrative probes, not validated labels or perfectly controlled lexical matches. Review them before interpreting results.

In [ ]:
tests=[
('P02','these mountains that you are holding,\nyou were only supposed to climb.','Challenges are obstacles to overcome rather than burdens to carry indefinitely.','These mountains are difficult to climb while holding heavy equipment.'),
('P01','faces are only geometry.\nlight resting temporarily on bone.','Physical appearance alone cannot capture inner experience.','Facial geometry and bone structure determine how light falls across a portrait.'),
('P04','broken glass reflects more light\nthan something whole ever could.','Hardship can lead to greater strength and complexity.','Broken glass reflects light at different angles from a whole glass panel.'),
('P03','the river doesn’t rush all the time—\nsometimes it drifts,\nsometimes it rests,\nbut it always flows.\n\nso do you.','Personal growth continues even during periods of apparent inactivity.','A river sometimes flows rapidly and sometimes drifts slowly depending on its surroundings.'),
('P08','ash is not the end.\nit is memory—','The aftermath of hardship retains traces of experiences that transformed us.','Ash is the residue left after material burns.'),
('P09','and when they pulled\nher grief away,\nher arms were shaped\nfor him to stay.','Absence does not erase the habits and attachments formed by love.','Her arms retained their shape after the object was pulled away.')]
probe_df=pd.DataFrame(tests,columns=['poem_id','original','meaning','literal']); display(probe_df)
results=[]
for name,m in models.items():
    for pid,o,a,b in tests:
        v=[embed(t,m)[0] for t in [o,a,b]]
        sa,sb=float(v[0]@v[1]),float(v[0]@v[2])
        results.append(dict(model=name,poem_id=pid,meaning_similarity=sa,literal_similarity=sb,margin=sa-sb,meaning_preferred=sa>sb))
probes=pd.DataFrame(results); display(probes); display(probes.groupby('model').meaning_preferred.agg(['sum','count']))
probes.to_csv(OUT/'metaphor_results.csv',index=False); probe_df.to_csv(OUT/'metaphor_prompts.csv',index=False)
plt.figure(figsize=(10,5)); sns.barplot(data=probes,x='poem_id',y='margin',hue='model'); plt.axhline(0,color='gray'); plt.ylabel('Meaning similarity − literal similarity'); plt.tight_layout(); plt.savefig(OUT/'metaphor_margins.png',dpi=220); plt.show(); plt.close()


## 6. Semantic search
AI-drafted queries and candidate targets are exploratory. BGE uses its recommended retrieval query prefix; other models use raw queries. Multiple poems may be valid answers.

In [ ]:
queries=[('Missing someone whose presence still shapes ordinary objects',['P01','P09']),('Feeling guilty about resting instead of making progress',['P03','P02']),('Finding understanding by slowing down and paying attention',['P05','P11']),('Surviving hardship without wanting applause',['P08','P04']),('Refusing to participate in an institution that profits from suffering',['P06','P07'])]
search=[]
for name,m in models.items():
    for q,targets in queries:
        prefix='Represent this sentence for searching relevant passages: ' if name=='BGE' else ''
        v=embed(q,m,prefix)[0]; scores=vectors[name]@v
        for rank,i in enumerate(np.argsort(-scores)[:3],1):
            search.append(dict(model=name,query=q,rank=rank,poem_id=ids[i],title=titles[i],similarity=float(scores[i]),draft_target=ids[i] in targets))
search_df=pd.DataFrame(search); display(search_df); search_df.to_csv(OUT/'semantic_search.csv',index=False)


## 7. Export results
Download the ZIP and send it back for interpretation. It excludes the full CSV corpus, but includes excerpts, titles and embeddings; treat it as private. No model runs have been performed in advance. Avoid generalizing from 12 poems or treating AI annotations as author ground truth.

In [ ]:
(OUT/'run_metadata.json').write_text(json.dumps(meta,indent=2),encoding='utf-8')
pd.DataFrame({'poem_id':ids,'title':titles}).to_csv(OUT/'poem_index.csv',index=False)
archive=shutil.make_archive('poetry_results','zip',OUT)
from google.colab import files
files.download(archive)
